# Day 14 · Job Task 2 — Aggregate

**Databricks + Snowflake · 70-Hour Programme · DataTrends.tech**

---

This task runs **only after** Task 1 finishes successfully. That is not politeness — it is
a rule you draw in the job, and Databricks enforces it.

The next cell creates two input boxes. They appear at the **top of the notebook**, not
underneath the cell.

In [ ]:
dbutils.widgets.text("my_id",   "",          "1 - Your MY_ID")
dbutils.widgets.text("catalog", "workspace", "2 - Catalog")

# The boxes appear at the TOP of the notebook, not below this cell.
print("Two boxes have been created at the TOP of this notebook.")
print("Scroll up, put your MY_ID in box 1, then run the next cell.")

In [ ]:
import re
from pyspark.sql import functions as F

MY_ID = dbutils.widgets.get("my_id").strip()
if not MY_ID:
    raise ValueError("Parameter my_id is empty. Set it on the task, or in the widget above.")

TAG     = re.sub(r"[^a-z0-9]+", "_", MY_ID.lower()).strip("_")
CATALOG = dbutils.widgets.get("catalog").strip() or "workspace"
SCHEMA  = f"day1213_{TAG}"

BRONZE  = f"{CATALOG}.{SCHEMA}.payments_bronze"
CLEAN   = f"{CATALOG}.{SCHEMA}.payments_clean"
SILVER  = f"{CATALOG}.{SCHEMA}.city_totals"

print(f"reading : {BRONZE}")

## What Task 1 left behind

`debugValue` is what you get when you run this notebook by hand, outside a job. Without it
the cell fails in the notebook and works only in the job, which is a miserable way to
develop.

In [ ]:
new_rows = dbutils.jobs.taskValues.get(taskKey="ingest", key="new_rows", debugValue=0)

print(f"Task 1 reported {new_rows} new rows")

if new_rows == 0:
    print("Nothing new arrived. The aggregate is rebuilt anyway — it is cheap and it is safe.")

## Rebuild clean and the totals

In [ ]:
(spark.read.table(BRONZE)
      .where("status = 'SUCCESS'")
      .write.mode("overwrite").saveAsTable(CLEAN))

(spark.read.table(CLEAN)
      .groupBy("city")
      .agg(F.sum("amount").alias("total_amount"),
           F.count("*").alias("txns"))
      .write.mode("overwrite").saveAsTable(SILVER))

display(spark.table(SILVER).orderBy(F.desc("total_amount")))

In [ ]:
total = spark.sql(f"SELECT sum(total_amount) AS t FROM {SILVER}").first()["t"]

print(f"total collected : {total}")

dbutils.jobs.taskValues.set(key="total_collected", value=int(total or 0))

## Why re-running this is safe

Task 1 is safe to repeat because the checkpoint remembers which files are done.
Task 2 is safe to repeat because it **overwrites** rather than appends — running it twice
gives the same answer as running it once.

A step that gives the same result however many times you run it is called **idempotent**.
It is the reason a job is allowed to retry itself at two in the morning without asking you.